# 07 — TMDB Keyword Valence

Matches all 84K TMDB keywords against the joined SCL lexicon (from `06_scl_join`)
using the OPP → NMA → VAD hierarchy, then computes per-keyword valence features
including token-level SCL coverage, modifier flags, and composition metadata.

## Matching strategy

```
1. Exact phrase match against joined SCL lexicon (OPP > NMA > VAD already resolved into `valence`)
2. Token-mean fallback (source = `computed`): average the hierarchy-selected `valence` of each unigram token
```

The token-mean fallback uses `valence` (not raw `vad_valence`), so OPP/NMA scores
take precedence even in the fallback.

## Output columns (per keyword)

| column | description |
|--------|-------------|
| `keyword` | TMDB keyword name |
| `ngrams` | word count |
| `in_scl` | True if the keyword is an exact phrase in the SCL joined lexicon |
| `scl_token_coverage` | fraction of keyword tokens found in SCL unigrams (0.0–1.0) |
| `valence` | best available valence score, bipolar [-1, +1] |
| `valence_scale` | `bipolar` when valence is not null (all sources use BWS [-1,+1]) |
| `valence_source` | `opp` / `nma` / `vad` (direct SCL match) or `computed` (token mean) |
| `valence_count` | number of lexicons with a score (from joined; 0 if computed) |
| `valence_variance` | variance across lexicon scores (null if < 2 sources) |
| `matched_tokens` | tokens with a valence score (for `computed` source) |
| `total_tokens` | total tokens in keyword |
| `negators_contained` | negator tokens found in keyword (e.g. `not`, `never`, `no`) |
| `modals_contained` | modal tokens found in keyword (e.g. `can`, `should`, `might`) |
| `adverbs_contained` | intensifier/diminisher adverbs found in keyword (e.g. `very`, `barely`) |
| `has_nma` | True if any negator, modal, or adverb is present |

## Aggregate features (per movie's keyword set)

```
matched_keyword_count   — keywords with any valence score
avg_keyword_valence     — mean valence across matched keywords
min_keyword_valence     — most negative keyword
max_keyword_valence     — most positive keyword
valence_std             — spread of sentiment across keywords
```

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('data')

## Load joined lexicon and TMDB keywords

In [2]:
lexicon = pd.read_pickle(DATA / 'lexicons/scl_joined.pkl')

# deduplicate: keep first occurrence per term
# (valence_source priority OPP > NMA > VAD is already baked into `valence`)
lex_dedup = (
    lexicon[['term', 'valence', 'valence_source', 'valence_count', 'valence_variance',
             'negators_contained', 'modals_contained', 'adverbs_contained']]
    .dropna(subset=['term'])
    .drop_duplicates(subset='term', keep='first')
)
print(f'Lexicon after dedup: {len(lex_dedup):,} unique terms (from {len(lexicon):,})')

# fast phrase lookup (includes modifier columns)
lex_lookup = lex_dedup.set_index('term').to_dict('index')

# unigram lookup using hierarchy-selected `valence` (not raw vad_valence)
# so OPP/NMA scores take precedence even in token-mean fallback
unigram_lookup = (
    lex_dedup[lex_dedup['term'].str.split().str.len() == 1]
    .set_index('term')['valence']
    .dropna()
    .to_dict()
)

# modifier token sets — derived from modifier columns in the SCL joined lexicon
# used for detecting NMA context in token-mean (computed) keywords
def _flatten_tokens(col):
    tokens = set()
    for v in lex_dedup[col].dropna():
        if isinstance(v, str) and v.strip():
            tokens.update(v.strip().split())
    return tokens

NEGATOR_TOKENS = _flatten_tokens('negators_contained')
MODAL_TOKENS   = _flatten_tokens('modals_contained')
ADVERB_TOKENS  = _flatten_tokens('adverbs_contained')

print(f'Phrase lookup entries:              {len(lex_lookup):,}')
print(f'Unigram entries for token fallback: {len(unigram_lookup):,}')
print(f'Negator tokens: {sorted(NEGATOR_TOKENS)}')
print(f'Modal tokens:   {sorted(MODAL_TOKENS)}')
print(f'Adverb tokens:  {sorted(ADVERB_TOKENS)}')

Lexicon after dedup: 57,001 unique terms (from 57,005)
Phrase lookup entries:              57,001
Unigram entries for token fallback: 44,811
Negator tokens: ['cannot', 'could', 'did', 'does', 'had', 'have', 'may', 'never', 'no', 'not', 'nothing', 'was', 'will', 'would']
Modal tokens:   ['can', 'could', 'may', 'might', 'must', 'should', 'would']
Adverb tokens:  ['certainly', 'especially', 'extremely', 'fairly', 'highly', 'increasingly', 'less', 'more', 'most', 'much', 'particularly', 'pretty', 'probably', 'quite', 'rather', 'really', 'relatively', 'so', 'too', 'very']


In [3]:
keywords = pd.read_csv(DATA / 'tmdb_keywords_canonical.csv')
print(f'TMDB keywords: {len(keywords):,}')
keywords.head(5)

TMDB keywords: 84,765


,tmdb_keyword_id,name
0,30,individual
1,65,holiday
2,74,germany
3,75,gunslinger
4,83,saving the world


## Score each keyword

In [4]:
def _detect_modifiers(tokens):
    """Detect NMA modifier tokens in a list of word tokens."""
    negators = [t for t in tokens if t in NEGATOR_TOKENS]
    modals   = [t for t in tokens if t in MODAL_TOKENS]
    adverbs  = [t for t in tokens if t in ADVERB_TOKENS]
    return negators, modals, adverbs


def _parse_modifier_col(val):
    """Parse a modifier column value (string token or NaN) into a list."""
    if not isinstance(val, str) or not val.strip():
        return []
    # single token stored as bare string, not list literal
    return val.strip().split()


def score_keyword(name):
    key = str(name).lower().strip()
    tokens = key.split()
    n = len(tokens)

    # token coverage in SCL unigrams
    scl_token_count = sum(1 for t in tokens if t in unigram_lookup)
    scl_token_coverage = scl_token_count / n if n > 0 else 0.0

    # 1. exact phrase match — valence already hierarchy-resolved (OPP > NMA > VAD)
    if key in lex_lookup:
        row = lex_lookup[key]
        negators = _parse_modifier_col(row.get('negators_contained'))
        modals   = _parse_modifier_col(row.get('modals_contained'))
        adverbs  = _parse_modifier_col(row.get('adverbs_contained'))
        return {
            'in_scl':              True,
            'scl_token_coverage':  scl_token_coverage,
            'valence':             row['valence'],
            'valence_scale':       'bipolar' if row['valence'] is not None and not (isinstance(row['valence'], float) and np.isnan(row['valence'])) else None,
            'valence_source':      row['valence_source'],
            'valence_count':       row['valence_count'],
            'valence_variance':    row['valence_variance'],
            'matched_tokens':      n,
            'total_tokens':        n,
            'negators_contained':  negators,
            'modals_contained':    modals,
            'adverbs_contained':   adverbs,
            'has_nma':             bool(negators or modals or adverbs),
        }

    # 2. token-mean fallback using hierarchy-selected unigram valence
    token_scores = [unigram_lookup[t] for t in tokens if t in unigram_lookup]
    negators, modals, adverbs = _detect_modifiers(tokens)
    if token_scores:
        return {
            'in_scl':              False,
            'scl_token_coverage':  scl_token_coverage,
            'valence':             float(np.mean(token_scores)),
            'valence_scale':       'bipolar',
            'valence_source':      'computed',
            'valence_count':       0,
            'valence_variance':    float(np.var(token_scores, ddof=1)) if len(token_scores) > 1 else None,
            'matched_tokens':      len(token_scores),
            'total_tokens':        n,
            'negators_contained':  negators,
            'modals_contained':    modals,
            'adverbs_contained':   adverbs,
            'has_nma':             bool(negators or modals or adverbs),
        }

    # 3. no match
    return {
        'in_scl':              False,
        'scl_token_coverage':  scl_token_coverage,
        'valence':             None,
        'valence_scale':       None,
        'valence_source':      None,
        'valence_count':       0,
        'valence_variance':    None,
        'matched_tokens':      0,
        'total_tokens':        n,
        'negators_contained':  negators,
        'modals_contained':    modals,
        'adverbs_contained':   adverbs,
        'has_nma':             bool(negators or modals or adverbs),
    }

scores = keywords['name'].apply(score_keyword).apply(pd.Series)
kw_scored = pd.concat([keywords, scores], axis=1)
kw_scored['ngrams'] = kw_scored['name'].str.split().str.len()

kw_scored = kw_scored[[
    'tmdb_keyword_id', 'name', 'ngrams',
    'in_scl', 'scl_token_coverage',
    'valence', 'valence_scale', 'valence_source', 'valence_count', 'valence_variance',
    'matched_tokens', 'total_tokens',
    'negators_contained', 'modals_contained', 'adverbs_contained', 'has_nma',
]]

print(f'Scored: {len(kw_scored):,} keywords')
print(f'\nin_scl breakdown:')
print(kw_scored['in_scl'].value_counts(dropna=False).to_string())
print(f'\nvalence_source breakdown:')
print(kw_scored['valence_source'].value_counts(dropna=False).to_string())
print(f'\nCoverage: {kw_scored["valence"].notna().sum():,} / {len(kw_scored):,} ({kw_scored["valence"].notna().mean()*100:.1f}%)')
print(f'\nhas_nma: {kw_scored["has_nma"].sum():,} keywords ({kw_scored["has_nma"].mean()*100:.1f}%)')
kw_scored.head(5)

Scored: 84,765 keywords

in_scl breakdown:
in_scl
False    72105
True     12660

valence_source breakdown:
valence_source
computed    36425
NaN         35680
vad         11523
nma           711
opp           426

Coverage: 49,085 / 84,765 (57.9%)

has_nma: 321 keywords (0.4%)


,tmdb_keyword_id,name,ngrams,in_scl,scl_token_coverage,valence,valence_scale,valence_source,valence_count,valence_variance,matched_tokens,total_tokens,negators_contained,modals_contained,adverbs_contained,has_nma
0,30,individual,1.0,True,1.000000,0.254,bipolar,vad,1,NaN,1,1,[],[],[],False
1,65,holiday,1.0,True,1.000000,0.656,bipolar,opp,2,0.028800,1,1,[],[],[],False
2,74,germany,1.0,False,0.000000,NaN,NaN,NaN,0,NaN,0,1,[],[],[],False
3,75,gunslinger,1.0,True,1.000000,-0.368,bipolar,vad,1,NaN,1,1,[],[],[],False
4,83,saving the world,3.0,False,0.666667,0.327,bipolar,computed,0,0.000002,2,3,[],[],[],False


## SCL token coverage

In [5]:
# Distribution of token coverage for unmatched (computed) keywords
computed = kw_scored[kw_scored['valence_source'] == 'computed']
unmatched = kw_scored[kw_scored['valence'].isna()]

print('Token coverage distribution (computed keywords):')
print(computed['scl_token_coverage'].describe().round(3).to_string())

print(f'\nUnmatched keywords (no valence): {len(unmatched):,}')
print('Token coverage for unmatched keywords:')
print(unmatched['scl_token_coverage'].describe().round(3).to_string())

print(f'\nUnmatched with at least 1 token in SCL: {(unmatched["scl_token_coverage"] > 0).sum():,}')
print(f'Unmatched with 0 token coverage:         {(unmatched["scl_token_coverage"] == 0).sum():,}')

print(f'\nKeywords with NMAs breakdown:')
print(kw_scored.groupby('has_nma')['valence_source'].value_counts(dropna=False).to_string())

Token coverage distribution (computed keywords):
count    36425.000
mean         0.795
std          0.248
min          0.048
25%          0.500
50%          1.000
75%          1.000
max          1.000

Unmatched keywords (no valence): 35,680
Token coverage for unmatched keywords:
count    35680.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0

Unmatched with at least 1 token in SCL: 0
Unmatched with 0 token coverage:         35,680

Keywords with NMAs breakdown:
has_nma  valence_source
False    computed          36168
         NaN               35635
         vad               11512
         nma                 706
         opp                 423
True     computed            257
         NaN                  45
         vad                  11
         nma                   5
         opp                   3


## Valence distribution

In [6]:
matched = kw_scored[kw_scored['valence'].notna()]

print('Valence stats (matched keywords only):')
print(matched['valence'].describe().round(4).to_string())

print(f'\nPositive (>0):  {(matched["valence"] > 0).sum():,}  ({(matched["valence"] > 0).mean()*100:.1f}%)')
print(f'Neutral  (=0):  {(matched["valence"] == 0).sum():,}  ({(matched["valence"] == 0).mean()*100:.1f}%)')
print(f'Negative (<0):  {(matched["valence"] < 0).sum():,}  ({(matched["valence"] < 0).mean()*100:.1f}%)')

print(f'\nMost positive keywords:')
print(matched.nlargest(10, 'valence')[['name', 'valence', 'valence_source', 'in_scl']].to_string(index=False))

print(f'\nMost negative keywords:')
print(matched.nsmallest(10, 'valence')[['name', 'valence', 'valence_source', 'in_scl']].to_string(index=False))

Valence stats (matched keywords only):
count    49085.0000
mean         0.0951
std          0.3829
min         -1.0000
25%         -0.1255
50%          0.1290
75%          0.3600
max          1.0000

Positive (>0):  30,882  (62.9%)
Neutral  (=0):  1,279  (2.6%)
Negative (<0):  16,924  (34.5%)

Most positive keywords:
           name  valence valence_source  in_scl
        sapient      1.0            vad    True
    empowerment      1.0            vad    True
       pacifism      1.0            vad    True
suburbian idyll      1.0       computed   False
   mercifulness      1.0            vad    True
   thankfulness      1.0            vad    True
      precocity      1.0            vad    True
         tahiti      1.0            vad    True
 mahatma gandhi      1.0       computed   False
invulnerability      1.0            vad    True

Most negative keywords:
            name  valence valence_source  in_scl
    ku klux klan     -1.0       computed   False
     nuclear war     -1.0     

## Aggregate features

Per movie: average valence, spread, and count across its keyword set.
Demonstrated on a simulated example keyword set.

In [7]:
print("""
For a movie M with keywords K = {k1, k2, ..., kn} where valence(ki) is defined:

  matched_keyword_count = |{ki : valence(ki) is not null}|
  avg_keyword_valence   = mean(valence(ki) for matched ki)
  min_keyword_valence   = min(valence(ki) for matched ki)
  max_keyword_valence   = max(valence(ki) for matched ki)
  valence_std           = std(valence(ki) for matched ki, ddof=1)

valence_source precedence: opp > nma > vad > computed (token mean)
""")

example_keywords = ['dark comedy', 'tragic love', 'happy ending', 'serial killer', 'redemption']
example_rows = []
for kw in example_keywords:
    s = score_keyword(kw)
    example_rows.append({'keyword': kw, **s})

ex = pd.DataFrame(example_rows)
matched_ex = ex[ex['valence'].notna()]

print('Example keyword set:')
print(ex[['keyword', 'valence', 'valence_source', 'in_scl', 'has_nma']].to_string(index=False))
print(f'\nmatched_keyword_count : {len(matched_ex)}')
print(f'avg_keyword_valence   : {matched_ex["valence"].mean():.4f}')
print(f'min_keyword_valence   : {matched_ex["valence"].min():.4f}')
print(f'max_keyword_valence   : {matched_ex["valence"].max():.4f}')
print(f'valence_std           : {matched_ex["valence"].std(ddof=1):.4f}')


For a movie M with keywords K = {k1, k2, ..., kn} where valence(ki) is defined:

  matched_keyword_count = |{ki : valence(ki) is not null}|
  avg_keyword_valence   = mean(valence(ki) for matched ki)
  min_keyword_valence   = min(valence(ki) for matched ki)
  max_keyword_valence   = max(valence(ki) for matched ki)
  valence_std           = std(valence(ki) for matched ki, ddof=1)

valence_source precedence: opp > nma > vad > computed (token mean)

Example keyword set:
      keyword  valence valence_source  in_scl  has_nma
  dark comedy  -0.1080            vad    True    False
  tragic love  -0.0085       computed   False    False
 happy ending   0.9000            vad    True    False
serial killer  -0.6420            vad    True    False
   redemption  -0.2480            vad    True    False

matched_keyword_count : 5
avg_keyword_valence   : -0.0213
min_keyword_valence   : -0.6420
max_keyword_valence   : 0.9000
valence_std           : 0.5686


## Save

In [8]:
out = DATA / 'lexicons/tmdb_keyword_valence.pkl'
kw_scored.to_pickle(out)
print(f'Saved → {out}  ({out.stat().st_size / 1e6:.1f} MB)')
print(f'Shape: {kw_scored.shape}')
print(f'Columns: {list(kw_scored.columns)}')

Saved → data/lexicons/tmdb_keyword_valence.pkl  (8.5 MB)
Shape: (84765, 16)
Columns: ['tmdb_keyword_id', 'name', 'ngrams', 'in_scl', 'scl_token_coverage', 'valence', 'valence_scale', 'valence_source', 'valence_count', 'valence_variance', 'matched_tokens', 'total_tokens', 'negators_contained', 'modals_contained', 'adverbs_contained', 'has_nma']
